# 02 — XGBoost and model selection
Tune XGBoost across all four forecast setups, then freeze each RF/XGBoost winner using validation MAE.
Final fits use training plus validation; test scores never change model selection.
Export MAE, RMSE, R² and per-row predictions for combined reporting.


In [ ]:
%pip -q install numpy pandas scikit-learn joblib optuna xgboost


In [ ]:
from pathlib import Path
import sys
import importlib
from urllib.request import urlopen

helper_candidates = [Path.cwd() / "kc_common.py", Path.cwd() / "KC_CO2_Modular_Notebooks" / "kc_common.py"]
helper_path = next((p for p in helper_candidates if p.is_file()), None)
if helper_path is None:
    HELPER_REF = "main"
    helper_url = f"https://raw.githubusercontent.com/Namelessr679/ESP32-Indoor-Air-Quality-1.0Monitor/{HELPER_REF}/KC_CO2_Modular_Notebooks/kc_common.py"
    try:
        with urlopen(helper_url, timeout=30) as response:
            helper_source = response.read()
    except Exception as exc:
        raise RuntimeError("kc_common.py is unavailable. Place the matching helper beside the notebook or select a published HELPER_REF, then rerun setup.") from exc
    compile(helper_source, "kc_common.py", "exec")
    if b"def prepare_experiment(" not in helper_source:
        raise RuntimeError("The downloaded helper is incompatible. Use kc_common.py from the same repository revision as these notebooks.")
    helper_path = Path.cwd() / "kc_common.py"
    helper_path.write_bytes(helper_source)
sys.path.insert(0, str(helper_path.parent))
import kc_common
if Path(kc_common.__file__).resolve() != helper_path.resolve():
    raise RuntimeError("A different kc_common module is already loaded. Restart the runtime and rerun setup.")
importlib.reload(kc_common)
from kc_common import experiment_run_dir, load_prepared, run_tree_configurations, validation_selections, export_tree_results


In [ ]:
from google.colab import drive
import joblib

drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/KC_CO2_Modular_Notebooks")
RUN_DIR = experiment_run_dir(PROJECT)


In [ ]:
prepared = load_prepared(RUN_DIR)


In [ ]:
def suggest_xgb_params(trial):
    params = {
        # Try different numbers of decision trees (100 to 500 in steps of 50)
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, step=50),
        # Try tree depths from 2 (very simple) up to 6 (more detailed/complex)
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        # The learning rate or step size shrinkage used to prevent overfitting (small steps are safer)
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        # Minimum sum of instance weight needed in a child node (helps prevent building overly specific rules)
        "min_child_weight": trial.suggest_float("min_child_weight", 1.0, 10.0, log=True),
        # Percentage of randomly chosen data rows to use when building each tree
        "subsample": trial.suggest_float("subsample", 0.65, 1.0),
        # Percentage of randomly chosen features to use when building each tree
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.55, 1.0),
        # Minimum loss reduction required to make a split (larger values prevent shallow splits)
        "gamma": trial.suggest_float("gamma", 0.0, 6.0),
        # L1 regularization term on weights (encourages weight values to go to zero)
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 3.0, log=True),
        # L2 regularization term on weights (smooths out weight changes)
        "reg_lambda": trial.suggest_float("reg_lambda", 0.2, 15.0, log=True),
    }
    return params


In [ ]:
rf_saved = joblib.load(RUN_DIR / "rf_validation.joblib")
if rf_saved["metadata"] != prepared["metadata"]:
    raise ValueError("RF results do not match preparation. Rerun notebook 01.")
rf_results = rf_saved["configurations"]
xgb_results = run_tree_configurations("XGBoost", suggest_xgb_params, prepared, RUN_DIR / "xgb_validation.joblib")
selected_models = validation_selections(rf_results, xgb_results)
print("Selected on validation:", selected_models)


## Final fitting, evaluation and export


In [ ]:
tree_results = export_tree_results(prepared, rf_results, xgb_results, selected_models)
if joblib.load(RUN_DIR / "preparation_metadata.joblib") != prepared["metadata"]:
    raise ValueError("Preparation changed during this run. Rerun 01 and 02.")
joblib.dump(tree_results, RUN_DIR / "tree_results.joblib", compress=3)
print(tree_results["metrics"].to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("Results saved for notebook 05.")
